# Analysis of Muon Data across Temperatures

### Importing Libraries

In [1]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd
from pathlib import Path
import linecache
from tqdm.notebook import tqdm
from dataclasses import dataclass                                                                                                       
from datetime import datetime, timezone, timedelta                                                                                                         
import bisect          
import requests 
import csv
from zoneinfo import ZoneInfo   
import re    
import math       
import os
from scipy.signal import find_peaks
                                                                                                                                                                                                                                                                                                                                                                  

### Temperature Helper Functions

In [2]:
@dataclass                                                                                                                                
class TemperatureLog:                                                                                                                     
    times: list[datetime]                                                                                                                 
    channels: dict[str, list[float | None]]                                                                                               
                                                                                                                                        
    @classmethod                                                                                                                          
    def from_response(cls, response):                                                                                                     
        data = response.json()                                                                                                            
                                                                                                                                        
        times = [                                                                                                                         
            datetime.fromisoformat(t)                                                                                                     
            for t in data["time"]                                                                                                         
        ]                                                                                                                                 
                                                                                                                                        
        channels = {                                                                                                                      
            name: values                                                                                                                  
            for name, values in data.items()                                                                                              
            if name != "time"                                                                                                             
        }                                                                                                                                 
                                                                                                                                        
        return cls(times, channels)                                                                                                       
                                                                                                                                        
    def at(self, time: datetime, channel: str = "MC_Sample"):                                                                             
        if not isinstance(time, datetime):                                                                                                
            raise TypeError("time must be a datetime.datetime object")                                                                    
                                                                                                                                        
        i = bisect.bisect_left(self.times, time)                                                                                          
                                                                                                                                        
        if i == 0:                                                                                                                        
            idx = 0                                                                                                                       
        elif i == len(self.times):                                                                                                        
            idx = len(self.times) - 1                                                                                                     
        else:                                                                                                                             
            before = self.times[i - 1]                                                                                                    
            after = self.times[i]                                                                                                         
            idx = i - 1 if abs(time - before) <= abs(after - time) else i                                                                 
                                                                                                                                        
        return self.channels[channel][idx]

def get_temperature(trigger_time, is_offset = True, channel="MC_Sample", window_seconds=120):
    if isinstance(trigger_time, str):
        trigger_time = datetime.strptime(
            trigger_time, "%Y-%m-%d %H:%M:%S"
        )

    if not isinstance(trigger_time, datetime):
        raise TypeError("trigger_time must be a datetime or YYYY-MM-DD HH:MM:SS string")

    if trigger_time.tzinfo is None:
        trigger_time = trigger_time.replace(
            tzinfo=ZoneInfo("Australia/Sydney")
        )

    if is_offset:
        offset = timedelta(hours=17, minutes=3, seconds=31)
        trigger_time = trigger_time + offset

    start = trigger_time - timedelta(seconds=window_seconds)
    stop = trigger_time + timedelta(seconds=window_seconds)

    response = requests.get(
        "https://qsyd.sydney.edu.au/data/Blue_Fridge",
        params={
            "start": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "stop": stop.strftime("%Y-%m-%dT%H:%M:%S"),
        },
    )
    response.raise_for_status()

    temperatures = TemperatureLog.from_response(response)

    if not temperatures.times:
        raise ValueError("No temperature measurements found near the trigger time")

    if not temperatures.channels.get(channel):
        raise ValueError(f"No measurements found for channel {channel!r}")

    return temperatures.at(trigger_time, channel=channel)


### Helper Functions

In [3]:
def file_reader(file, inv_amp = True):
    filename = Path(file).name

    # extract timestamps such as: SaveOnEvent_ALL_20260904064221341.csv YYYYMMDDHHMMSSmmm
    match = re.search(r"(?<!\d)(\d{14})(\d{1,6})?(?!\d)", filename)
    if not match:
        raise ValueError(f"No timestamp found in filename: {file}")

    timestamp = datetime.strptime(match.group(1), "%Y%m%d%H%M%S")

    ch1_voltages, ch2_voltages = np.loadtxt(
        file, skiprows=17, usecols=(1, 2), delimiter=',', unpack=True
    )

    if inv_amp:
        ch1_voltages = -1*ch1_voltages
        ch2_voltages = -1*ch2_voltages

    return filename, timestamp, np.min(ch1_voltages), np.min(ch2_voltages)

def parse_directory(data_dir, output_csv):
    """
    Apply file_reader() to every CSV file in data_dir and write the results
    into one output CSV.

    file_reader() is expected to return:
        filename, timestamp, minimum_channel1, minimum_channel2
    """
    data_dir = Path(data_dir)
    output_csv = Path(output_csv)

    if not data_dir.is_dir():
        raise NotADirectoryError(f"Directory not found: {data_dir}")

    output_path = output_csv.resolve()
    csv_files = sorted(
        path
        for path in data_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".csv"
        and path.resolve() != output_path
    )

    processed = 0
    skipped = []

    with output_csv.open("w", newline="", encoding="utf-8") as output:
        writer = csv.writer(output)
        writer.writerow([
            "filename",
            "timestamp",
            "ch1_min_voltage",
            "ch2_min_voltage",
        ])

        for csv_path in csv_files:
            try:
                filename, timestamp, ch1_min, ch2_min = file_reader(csv_path, inv_amp=True)

                # Use the actual filename if file_reader returns None.
                if filename is None:
                    filename = csv_path.name

                if isinstance(timestamp, datetime):
                    timestamp = timestamp.isoformat(
                        sep=" ",
                        timespec="seconds",
                    )

                writer.writerow([
                    filename,
                    timestamp,
                    ch1_min,
                    ch2_min,
                ])

                processed += 1

            except Exception as error:
                skipped.append((csv_path.name, str(error)))
                print(f"Skipping {csv_path.name}: {error}")

    print(f"Processed: {processed} files")
    print(f"Skipped:   {len(skipped)} files")
    print(f"Output:    {output_csv}")

def find_coincidences(filenames, timestamp, ch1_min_voltages, ch2_min_voltages, threshold, save_coinfiles=False):
    # initialize counters
    coincidence_count = 0
    coin_filename = []

    # loop through the filenames and check for coincidences
    for i in range(len(filenames)):
        #convert min_voltage to float type
        ch1_min_voltage = float(ch1_min_voltages[i])
        ch2_min_voltage = float(ch2_min_voltages[i])

        # count coincidences given the threshold
        if ch1_min_voltage <= threshold and ch2_min_voltage <= threshold:
            coincidence_count += 1
            if save_coinfiles:
                coin_filename.append(filenames[i])

    # calculate duration of data taking
    first_timestamp = pd.to_datetime(timestamp[0])
    last_timestamp = pd.to_datetime(timestamp[-1])
    duration_hours = (last_timestamp - first_timestamp).total_seconds() / 3600

    # calculate rate
    coincidence_rate = coincidence_count / duration_hours if duration_hours > 0 else -1

    return coincidence_count, duration_hours, coincidence_rate

def hourly_coincidence_rate(filenames, timestamp, ch1_min_voltages, ch2_min_voltages, threshold = -0.005, is_timeoffset = True):

    filenames = np.asarray(filenames)
    original_trig_times = np.asarray(timestamp)
    parsed_trig_times = pd.to_datetime(original_trig_times, errors="raise")
    ch1_min_voltages = np.asarray(ch1_min_voltages).astype(float)
    ch2_min_voltages = np.asarray(ch2_min_voltages).astype(float)

    window_hours = 1
    offset = timedelta(hours=17, minutes=3, seconds=31)
    start_time = parsed_trig_times.min()
    experiment_end = parsed_trig_times.max()
    number_of_windows = int(np.floor((experiment_end - start_time)/pd.Timedelta(hours=window_hours)))

    rates = []
    labels = []
    temps = []

    for i in range(number_of_windows):
        window_start = start_time + pd.Timedelta(hours=i * window_hours)
        window_end = min(window_start + pd.Timedelta(hours=window_hours), experiment_end)

        in_window = ((parsed_trig_times >= window_start) & (parsed_trig_times < window_end))

        actual_window_hours = (window_end - window_start).total_seconds() / 3600

        if np.any(in_window):
            coincidence_count, _, _ = find_coincidences(
                filenames[in_window],
                original_trig_times[in_window],
                ch1_min_voltages[in_window],
                ch2_min_voltages[in_window],
                threshold
            )

            rate = coincidence_count / actual_window_hours
        else:
            rate = -1

        rates.append(rate)
        labels.append(
            f"{window_start + offset:%m-%d %H:%M}–"
            f"{window_end + offset:%H:%M}"
        )

        temps.append(get_temperature(window_start))

        print(f"Window {i + 1}: {window_start + offset} to {window_end + offset}, Coincidence Rate: {rate:.6f} coincidences/hour, Temperature: {temps[i]:.2f} mK")

    fig, ax_bottom = plt.subplots(figsize=(10, 6))

    ax_bottom.bar(labels, rates, color="steelblue", label="Coincidence rate")
    ax_bottom.set_xlabel("Hourly interval")
    ax_bottom.set_ylabel("Coincidence rate (coincidences/hour)")
    ax_bottom.set_title("Coincidence Rate Every Two Hours")
    plt.setp(ax_bottom.get_xticklabels(), rotation=90, ha="right")

    # Right-hand axis: temperature scale, scattered at the same x-position
    # (hourly interval) as each corresponding rate bar.
    ax_right = ax_bottom.twinx()
    ax_right.scatter(labels, temps, color="crimson", marker="o", zorder=3, label="Temperature")
    ax_right.set_ylabel("Temperature (mK)", color="crimson")
    ax_right.tick_params(axis="y", labelcolor="crimson")

    handles_bottom, labels_bottom = ax_bottom.get_legend_handles_labels()
    handles_right, labels_right = ax_right.get_legend_handles_labels()
    ax_bottom.legend(handles_bottom + handles_right, labels_bottom + labels_right, loc="upper right")

    fig.tight_layout()
    plt.show()

    return labels, rates, temps

def find_top_peaks(file, n_peaks=2, skiprows=17, time_col=0, value_col=1, min_separation_s=None):
    time = np.loadtxt(file, skiprows=skiprows, usecols=time_col, delimiter=',', dtype=float)
    voltage = np.loadtxt(file, skiprows=skiprows, usecols=value_col, delimiter=',', dtype=float)

    distance = None
    if min_separation_s is not None:
        dt = np.median(np.diff(time))
        distance = max(1, int(round(min_separation_s / dt)))

    peaks, props = find_peaks(voltage, prominence=0, distance=distance)

    if len(peaks) == 0:
        raise ValueError(f"No peaks found in {file}")
    if len(peaks) < n_peaks:
        raise ValueError(f"Only {len(peaks)} peak(s) found in {file}, need {n_peaks}")

    order = np.argsort(props["prominences"])[::-1][:n_peaks]
    top_idx = peaks[order]
    top_prom = props["prominences"][order]

    results = [
        {"index": int(i), "time": float(time[i]), "voltage": float(voltage[i]), "prominence": float(p)}
        for i, p in zip(top_idx, top_prom)
    ]
    results.sort(key=lambda r: r["time"])  # chronological order
    return results

def peak_stats_for_directory(data_dir, value_col=1, skiprows=17, min_separation_s=None):
    """
    Apply find_top_peaks() to every CSV in data_dir (single channel = value_col).

    Returns
    -------
    time_diffs : np.ndarray
        Δt between peak2 and peak1, one entry per successfully processed file.
    peak1_amps : np.ndarray
        Amplitude (voltage) of the first (earlier) peak.
    peak2_amps : np.ndarray
        Amplitude (voltage) of the second (later) peak.

    Files that error out (no timestamp needed here, but bad peak counts, etc.)
    are skipped and reported.
    """
    data_dir = Path(data_dir)
    if not data_dir.is_dir():
        raise NotADirectoryError(f"Directory not found: {data_dir}")

    csv_files = sorted(
        path for path in data_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".csv"
    )

    time_diffs = []
    peak1_amps = []
    peak2_amps = []
    skipped = []

    for csv_path in csv_files:
        try:
            p1, p2 = find_top_peaks(
                csv_path, n_peaks=2, skiprows=skiprows,
                value_col=value_col, min_separation_s=min_separation_s,
            )
            time_diffs.append(p2["time"] - p1["time"])
            peak1_amps.append(p1["voltage"])
            peak2_amps.append(p2["voltage"])
        except Exception as error:
            skipped.append((csv_path.name, str(error)))
            print(f"Skipping {csv_path.name}: {error}")

    print(f"Processed: {len(time_diffs)} files")
    print(f"Skipped:   {len(skipped)} files")

    return np.array(time_diffs), np.array(peak1_amps), np.array(peak2_amps)


def add_temperatures(input_csv: str, output_csv: str, channel: str = "MC_Sample"):
    tz = timezone(timedelta(hours=10))  # Sydney offset used elsewhere in your code

    # Read all rows first so we know the time range to fetch
    with open(input_csv, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    times = [
        datetime.strptime(row["trigTime"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=tz)
        for row in rows
    ]

    start = min(times) - timedelta(minutes=1)  # small padding
    stop = max(times) + timedelta(minutes=1)

    url = (
        "https://qsyd.sydney.edu.au/data/Blue_Fridge"
        f"?start={start.strftime('%Y-%m-%dT%H%%3A%M%%3A%S')}"
        f"&stop={stop.strftime('%Y-%m-%dT%H%%3A%M%%3A%S')}"
    )

    r = requests.get(url)
    temps = TemperatureLog.from_response(r)

    for row, t in zip(rows, times):
        row["temperature_mK"] = temps.at(t, channel=channel)

    with open(output_csv, "w", newline="") as f:
        fieldnames = list(rows[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


### Plotting Helper Functions

In [4]:
from matplotlib import legend


def plot_histogram(ch1_min_voltages, ch2_min_voltages, log_scale=False):
    ch1_min_voltages = np.array(ch1_min_voltages).astype(float)
    ch2_min_voltages = np.array(ch2_min_voltages).astype(float)

    # exclude infinite values
    ch1_min_voltages = ch1_min_voltages[np.isfinite(ch1_min_voltages)]
    ch2_min_voltages = ch2_min_voltages[np.isfinite(ch2_min_voltages)]
    
    plt.figure(figsize=(12, 6))
    plt.hist(ch1_min_voltages, bins=50, alpha=0.5, label='Channel 1', color='blue', log=log_scale)
    plt.hist(ch2_min_voltages, bins=50, alpha=0.5, label='Channel 2', color='red', log=log_scale)
    if log_scale:
        plt.title('Histogram of Minimum Voltages for Channels 1 and 2 (Log Scale)')
    else:
        plt.title('Histogram of Minimum Voltages for Channels 1 and 2')
    plt.xlabel('Minimum Voltage (mV)')
    plt.ylabel('Frequency')
    plt.legend()    
    plt.show()

def plot_waveforms(file):
    fig, ax = plt.subplots(2, 1)

    time = np.loadtxt(file, skiprows=17, usecols=0, delimiter=',', dtype=float)
    ch1_voltages = np.loadtxt(file, skiprows=17, usecols=1, delimiter=',', dtype=float)
    ch2_voltages = np.loadtxt(file, skiprows=17, usecols=2, delimiter=',', dtype=float)

    ax[0].plot(time, ch1_voltages, label='ch1', color='yellow')
    ax[1].plot(time, ch2_voltages, label='ch2', color='violet')

    ax[2].plot(time, ch1_voltages, label="ch1", color="yellow")
    ax[2].plot(time, ch2_voltages, label="ch2", color="violet")

    for axis in ax:
        axis.set_facecolor("black")
        axis.grid()
        axis.legend()
        axis.set_ylabel("Voltage (V)")

    ax[2].set_xlabel("Time (s)")

    fig.tight_layout()
    plt.show()

# plot time diff histogram 
def plot_histogram_2pks(time_diff, peak1, peak2, channel, log_scale=False):
    time_diff = np.array(time_diff).astype(float)
    peak1 = np.array(peak1).astype(float)
    peak2 = np.array(peak2).astype(float)


   # Keep only indices where both values are finite
    mask = np.isfinite(peak1) & np.isfinite(peak2)
    time_diff = time_diff[mask]
    peak1 = peak1[mask]
    peak2 = peak2[mask]
    
    n = len(time_diff)

    plt.figure(figsize=(12, 6))
    plt.hist(peak1, bins=50, alpha=0.5, label='Peak 1', color='blue', log=log_scale)
    plt.hist(peak2, bins=50, alpha=0.5, label='Peak 2', color='red', log=log_scale)
    if log_scale:
        plt.title(f'Histogram of Two Peaks (Log Scale) Channel {channel}')
    else:
        plt.title(f'Histogram of Two Peaks Channel {channel}')
    plt.xlabel('Maximum Amplitude (mV)')
    plt.ylabel('Frequency')
    plt.legend()    
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.hist(time_diff, bins=100*int(np.sqrt(n)), alpha=0.5, label='time_diff', color='blue', log=log_scale)
    if log_scale:
        plt.title(f'Histogram of Time Difference (Log Scale) Channel {channel}')
    else:
        plt.title(f'Histogram of Time Difference Channel {channel}')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency')
    plt.xscale('log')
    plt.legend()    
    plt.show()

# plot time diff histogram 
def plot_scatter_2pks(time_diff, peak1, peak2, channel):
    time_diff = np.array(time_diff).astype(float)
    peak1 = np.array(peak1).astype(float)
    peak2 = np.array(peak2).astype(float)

    # Keep only indices where both values are finite
    mask = np.isfinite(peak1) & np.isfinite(peak2)
    time_diff = time_diff[mask]
    peak1 = peak1[mask]
    peak2 = peak2[mask]

    ratio = peak2/peak1

    plt.figure(figsize=(12, 6))
    # plt.scatter(time_diff, peak1, alpha=0.5, color='blue', label="First Spike")
    plt.scatter(time_diff, peak2, alpha=0.5, color='red', label="Broad Pulse")
    plt.title(f'Peaks v/s Time Difference for Channel {channel}')
    plt.xlabel('Delta Time (s)')
    # plt.xscale('log')
    # plt.yscale('log')
    plt.ylabel('Peak Amplitudes (mV)')
    plt.legend()    
    plt.show()
    
    plt.figure(figsize=(12, 6))
    plt.scatter(peak1, peak2, alpha=0.5, color='blue')
    plt.title(f'Peak 2 v/s Peak 1 Channel {channel}')
    plt.xlabel('Peak1 (mV)')
    plt.xscale('log')
    plt.yscale('log')
    plt.ylabel('Peak2 (mV)')
    plt.legend()    
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.scatter(time_diff, ratio, alpha=0.5, color='blue')
    plt.title(f'A2/A1 v/s Time Difference Channel {channel}')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Time Difference (s)')
    plt.ylabel('A2/A1')
    plt.legend()    
    plt.show()


### Processing Raw Data into CSV File

In [5]:
data_dir = Path("/archive/4sepmdetectshigherupwkndrun")  # Update this path to your data directory
csv_file = Path("parsed_sep4weekend.csv")
parse_directory(data_dir, csv_file)

Skipping ._SaveOnEvent_ALL_20260904001131655.csv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte
Skipping ._SaveOnEvent_ALL_20260904041907300.csv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte
Skipping SaveOnEvent_ALL_20260904000836776.csv: zero-size array to reduction operation minimum which has no identity


/tmp/ipykernel_485953/2105708435.py:11: UserWarning: loadtxt: input contained no data: "/archive/4sepmdetectshigherupwkndrun/SaveOnEvent_ALL_20260904000836776.csv"
  ch1_voltages, ch2_voltages = np.loadtxt(


Processed: 662 files
Skipped:   3 files
Output:    parsed_sep4weekend.csv


### Calculating Coincidences

In [8]:
# make lists from each column of the csv file
with open(csv_file, "r") as f:
    lines = f.readlines()[1:]  # Skip header
    filename, timestamp, ch1_min_voltages, ch2_min_voltages = zip(*(line.strip().split(',') for line in lines))
 
coincidence_count, duration_hours, coincidence_rate = find_coincidences(filename, timestamp, ch1_min_voltages, ch2_min_voltages, threshold=-0.003, save_coinfiles=True)

print(f"Total Muon Coincidences: {coincidence_count}")
print(f"Duration: {duration_hours} hours")
print(f"Coincidence Rate: {coincidence_rate:.6f} coincidences per hour")
print()

clock_offset = timedelta(hours=17, minutes=3, seconds=31)
start_time = datetime.fromisoformat(timestamp[0]) + clock_offset
stop_time = datetime.fromisoformat(timestamp[-1]) + clock_offset

print(f"Start Time: {start_time}")
print(f"Start Temperature: {get_temperature(timestamp[0])} mK")
print()
print(f"Stop Time: {stop_time}")
print(f"Stop Temperature: {get_temperature(timestamp[-1])} mK")


Total Muon Coincidences: 487
Duration: 65.195 hours
Coincidence Rate: 7.469898 coincidences per hour

Start Time: 2026-09-04 17:15:02
Start Temperature: 47.45529778225724 mK

Stop Time: 2026-09-07 10:26:44
Stop Temperature: 44.56567817269787 mK


In [ ]:
hourly_coincidence_rate(filename, timestamp, ch1_min_voltages, ch2_min_voltages, threshold=-0.005, is_timeoffset = True)

### Time Difference

In [ ]:
ch1_time_diffs, ch1_peak1_amps, ch1_peak2_amps = peak_stats_for_directory(
    "/home/sandhya/Desktop/cryo-data/8sepheating4kupwards", value_col=1, min_separation_s=2e-7   # ch1; use value_col=2 for ch2
)


In [ ]:
ch2_time_diffs, ch2_peak1_amps, ch2_peak2_amps = peak_stats_for_directory(
    "/home/sandhya/Desktop/cryo-data/8sepheating4kupwards", value_col=2, min_separation_s=2e-7   # ch1; use value_col=2 for ch2
)

In [ ]:
print(np.mean(ch1_time_diffs))
print(np.mean(ch2_time_diffs))


In [ ]:
plot_scatter_2pks(ch1_time_diffs, ch1_peak1_amps, ch1_peak2_amps, 1)
plot_scatter_2pks(ch2_time_diffs, ch2_peak1_amps, ch2_peak2_amps, 2)

In [ ]:
plot_histogram_2pks(ch1_time_diffs, ch1_peak1_amps, ch1_peak2_amps, 1, log_scale=True)

In [ ]:
plot_histogram_2pks(ch2_time_diffs, ch2_peak1_amps, ch2_peak2_amps, 2, log_scale=True)

### Plotting Histograms

In [ ]:
# plotting histogram of min_voltages for both channels 
plot_histogram(ch1_min_voltages, ch2_min_voltages, log_scale=True)

### Checking Temperatures

In [ ]:
year = 2026
month = 8
day = 24
hour = 17
minute = 36
second = 27  

mytime = datetime(year, month, day, hour, minute, second,tzinfo=timezone(timedelta(hours=10)))

r = requests.get("https://qsyd.sydney.edu.au/data/Blue_Fridge?start=2026-09-08T09%3A47%3A04&stop=2026-09-08T09%3A47%3A05")
r = requests.get("https://qsyd.sydney.edu.au/data/Blue_Fridge?current")
# temps = TemperatureLog.from_response(r)

# temp = temps.at(mytime, channel="MC_Sample")

# print(f"Temperature at {mytime} is {temp} mK")

# print(get_temperature(trigTimes[0]))

r.content

In [ ]:
add_temperatures("parsed_results_sep7.csv", "sep7_with_temp.csv")

### Plotting Coincidence Waveforms

In [ ]:

# print(coin_files)

def plot_coincidence_waveforms(waveform_dir, coin_files):
    coin_files = [Path(waveform_dir) / filename for filename in coin_files]

    for file1, file2 in zip(coin_files[0::2], coin_files[1::2]):
        # print("File 1:", file1.name)
        # print("File 2:", file2.name)

        plot_waveforms(file1, file2)


plot_coincidence_waveforms(
    Path("27augmuon-1k-24.5v/"),
    coin_files
)

#